## Overview

This notebook is intended to:
* Introduce the core views and functions available in Awesense's Energy Data Model (EDM) SQL API.
* Demonstrate various ways to fetch and work with data using SQL and Python.

For complete documentation on the available views and functions, please refer to the [access_and_documentation.ipynb](../1_access_and_documentation/access_and_documentation.ipynb) notebook.

---

## Set up

In [1]:
import getpass
import os
from dotenv import load_dotenv
import psycopg2

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials, or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
# Checks for Google Colab: If detected,
# it bypasses reading credentials from local files and securely draws from Colab Secrets.
try:
    from google.colab import userdata
    from google.colab.userdata import SecretNotFoundError
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from contextlib import suppress
    print('☁️ Running in Google Colab. Using Colab Secrets for SQLAPI connection')
    # Look for Colab Secrets, fall back to interactive prompts if missing
    with suppress(SecretNotFoundError): os.environ['EDM_HOST'] = userdata.get('EDM_HOST')
    with suppress(SecretNotFoundError): os.environ['EDM_USER'] = userdata.get('EDM_USER')
    with suppress(SecretNotFoundError): os.environ['EDM_PASSWORD'] = userdata.get('EDM_PASSWORD')
else:
    # If running locally, loads connection parameters from a `.env` file (in this directory or parent directories)
    load_dotenv()

# Prompt the user to manually enter any missing connection parameters not found in `.env` file / Google Colab Secrets
if 'EDM_HOST' not in os.environ: os.environ['EDM_HOST'] = input('EDM server address: ').strip()
if 'EDM_USER' not in os.environ: os.environ['EDM_USER'] = input('EDM username: ')

# psycopg2/libpq automatically read the PG* environment variables, so map the shared EDM_* names onto them.
# (The password is only mapped if provided via `.env`/Colab; otherwise libpq falls back to `~/.pgpass`.)
os.environ['PGHOST'] = os.environ['EDM_HOST']
os.environ['PGUSER'] = os.environ['EDM_USER']
os.environ.setdefault('PGDATABASE', 'edm')
if 'EDM_PASSWORD' in os.environ: os.environ['PGPASSWORD'] = os.environ['EDM_PASSWORD']

print('Verifying SQLAPI connection parameters/credentials with connection attempt')

try:
    # Test database connection
    conn = psycopg2.connect('')
    conn.close()
    print('✅ SQLAPI connection parameters/credentials verified')
except psycopg2.OperationalError:
    print('⚠️ SQLAPI credentials check failed (assume missing/incorrect password, but double-check `.env` / secrets!)')
    # Prompt user securely for password
    os.environ['PGPASSWORD'] = getpass.getpass('Enter EDM Password manually: ')

# Keeps `%sql` result tables rendering correctly on newer versions of prettytable
import prettytable
if 'DEFAULT' not in vars(prettytable): prettytable.DEFAULT = prettytable.TableStyle.DEFAULT

# Load the SQL extension and connect
get_ipython().run_line_magic('reload_ext', 'sql')
get_ipython().run_line_magic('sql', 'postgresql://')

Verifying SQLAPI connection parameters/credentials with connection attempt


✅ SQLAPI connection parameters/credentials verified


---

## Views

**Grid**

The term `grid` represents a portion of a utility's grid, and the term `grid element` represents the physical elements that comprise the grid.

The view named `grid` lists available grids in the database, and `grid_id` is the primary key of the grid.

In [3]:
%%sql

SELECT *
FROM grid;

 * postgresql://


230 rows affected.


grid_id,description,last_updated
awefice,awefice,2021-01-01 20:00:00+00:00
Greensboro-Industrial,Greensboro-Industrial,2023-09-20 00:00:00+00:00
Greensboro-Industrial_missing,Greensboro-Industrial_missing,2023-09-20 00:00:00+00:00
Greensboro-Rural,Greensboro-Rural,2023-09-20 00:00:00+00:00
Greensboro-Rural_missing,Greensboro-Rural_missing,2023-09-20 00:00:00+00:00
Greensboro-Urban,Greensboro-Urban,2023-09-20 00:00:00+00:00
Greensboro-Urban_missing,Greensboro-Urban_missing,2023-09-20 00:00:00+00:00
GSO_1,GSO_1,2023-09-20 00:00:00+00:00
GSO_10,GSO_10,2023-09-20 00:00:00+00:00
GSO_100,GSO_100,2023-09-20 00:00:00+00:00


**Grid Element**

The `grid_element` view contains information about grid elements in a grid. Properties that are common to all grid elements, such as `phases`, are explicitly listed as columns of the view. And additional metadata that may differ between grid elements of different types (e.g. meter vs. transformers, etc.) are stored as JSONB in the `meta` column. 

The `grid_id` column from the `grid_element` view references the `grid_id` from the `grid` view. And the `grid_id` and `grid_element_id` columns are often used as input arguments to various functions, as demonstrated in the **Functions** section below and in the [grid_tracing.ipynb](../3_grid_tracing/grid_tracing.ipynb) notebook. 

In [4]:
%%sql

SELECT *
FROM grid_element
WHERE grid_id = 'awefice'
LIMIT 1;

 * postgresql://


1 rows affected.


grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,meta
awefice,batt_10,Battery,battery,ABC,None,True,True,False,None,con_135_t2,VCN.batt_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': 'batt_10', 'label': 'batt_10', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_10', 'longitude': None, 'lv_feeder': None, 'mv_feeder': None, 'ownership': 'private', 'make_model': 'LG CHEM RESU10H', 'enclosure_id': None, 'voltage_level': 'LV', 'capacity_usable': 9.6, 'commission_date': None, 'capacity_nominal': 9.8, 'manufacture_date': None, 'efficiency_round_trip': 97.96, 'max_peak_charge_power': 7.0, 'parent_transformer_id': 'transformer_36', 'max_peak_discharge_power': 7.0, 'max_continuous_charge_power': 5.0, 'max_continuous_discharge_power': 5.0}"


Below are different grid element types available in the `awefice` grid. 

A CSV file detailing all grid element types in the `Sandbox` server and their respective properties that the Awesense Platform recognizes can be found [here](https://sandbox.awesense.com/docs/_static/gis_metadata.csv).

In [5]:
%%sql

SELECT DISTINCT type as element_type
FROM grid_element
WHERE grid_id = 'awefice'
ORDER BY element_type;

 * postgresql://


15 rows affected.


element_type
ACLineSegment
Battery
Busbar
Cabinet
CircuitBreaker
Disconnector
EVCharger
Fuse
Jumper
Meter


A complete list of properties available in the `meta` column for a grid element can be returned using the built-in `json_each_text()` function. The following is an example of accessing each item in the `meta` column with the 'awefice' `grid_id` and the 'transormer_2' `grid_element_id`.

In [6]:
%%sql

SELECT ge.grid_element_id,
        meta_data.key,
        meta_data.value
FROM grid_element ge
JOIN json_each_text(ge.meta::json) AS meta_data
    ON true
WHERE grid_id = 'awefice'
    AND grid_element_id = 'transformer_2';

 * postgresql://


22 rows affected.


grid_element_id,key,value
transformer_2,name,tr_2
transformer_2,label,tr_2
transformer_2,latitude,None
transformer_2,load_loss,246932
transformer_2,longitude,None
transformer_2,mv_feeder,None
transformer_2,ownership,utility
transformer_2,make_model,None
transformer_2,rating_kva,20000.0
transformer_2,enclosure_id,enc_1


Specific properties from the `meta` column can be fetched individually as well.

In [7]:
%%sql

SELECT meta ->> 'rating_kva' as rating_kva,
       meta ->> 'commission_date' as commission_date
FROM grid_element
WHERE grid_id = 'awefice'
    AND grid_element_id = 'transformer_2';

 * postgresql://


1 rows affected.


rating_kva,commission_date
20000.0,1989-01-01


The `geometry` field can be easily converted to `latitude` and `longitude` values using postGIS functions.

For grid elements that are single points, such as `transformers`:

In [8]:
%%sql

SELECT ST_Y(geometry) as latitude,
        ST_X(geometry) as longitude
FROM grid_element
WHERE grid_id = 'awefice'
    AND grid_element_id = 'transformer_2';

 * postgresql://


1 rows affected.


latitude,longitude
49.26756406,-123.107138


For grid elements such as `line_segment` with starting and ending points, `geometry` can be used to display the `latitude` and `longitude` of the central point of the line as well as the lines' start and end coordinates:

In [9]:
%%sql

SELECT ST_Y(ST_Centroid(geometry)) as center_point_latitude,
    ST_X(ST_Centroid(geometry)) as center_point_longitude,
    ST_Y(ST_StartPoint(geometry)) as start_point_latitude,
    ST_X(ST_StartPoint(geometry)) as start_point_longitude,
    ST_Y(ST_EndPoint(geometry)) as end_point_latitude,
    ST_X(ST_EndPoint(geometry)) as end_point_longitude
FROM grid_element
WHERE grid_id = 'awefice'
    AND grid_element_id = 'line_segment_81';

 * postgresql://


1 rows affected.


center_point_latitude,center_point_longitude,start_point_latitude,start_point_longitude,end_point_latitude,end_point_longitude
49.26788015004806,-123.10579106630676,49.2678715685268,-123.105327920392,49.2679227270403,-123.106219178449


**Grid Element Data Source**

The `grid_element_data_source` view represents the linking between time-series data and physical elements on the grid. The primary key of this view is `grid_element_data_source_id`, and is used as an input argument of the `ts_data_source_select()` function to retrieve time series data, as demonstrated in the **Functions** section below and in the [time_series.ipynb](../4_time_series/time_series.ipynb) notebook.

In [10]:
%%sql

SELECT *
FROM grid_element_data_source
LIMIT 5;

 * postgresql://


5 rows affected.


grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction
9eadb399-7cf0-43f6-a6a9-c9758edb2b69,awefice,switchable_element_4,"[2021-01-01 08:01:00+00:00, None)",ABC,SENSOR,awe_scada_reads,['kWh'],scada_sensor_4,None
c8ca7a30-3025-41b2-b584-39e8500256d2,awefice,switchable_element_8,"[2021-01-01 08:01:00+00:00, None)",ABC,SENSOR,awe_scada_reads,['kWh'],scada_sensor_8,None
c1f0a432-771e-46fc-aa84-efef5dd84c5e,awefice,m_1,"[2021-01-01 08:01:00+00:00, None)",A,CONSUMER,awe_voltage_reads,['V'],m_1,None
0d019e82-de4a-4115-9c64-e8288ade79e9,awefice,m_2,"[2021-01-01 08:01:00+00:00, None)",A,CONSUMER,awe_voltage_reads,['V'],m_2,None
adc65088-66bd-49ba-9deb-a42e538a50a6,awefice,m_3,"[2021-01-01 08:01:00+00:00, None)",A,CONSUMER,awe_voltage_reads,['V'],m_3,None


The `type` field refers to the class of stored data, and there may be multiple `type` values for a given grid element. To retrieve the exact time series data of interest, filter the `grid_element_data_source` by the `type` as below.

In [11]:
%%sql

SELECT *
FROM grid_element_data_source
WHERE type = 'CONSUMER'
LIMIT 5;

 * postgresql://


5 rows affected.


grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction
c1f0a432-771e-46fc-aa84-efef5dd84c5e,awefice,m_1,"[2021-01-01 08:01:00+00:00, None)",A,CONSUMER,awe_voltage_reads,['V'],m_1,None
0d019e82-de4a-4115-9c64-e8288ade79e9,awefice,m_2,"[2021-01-01 08:01:00+00:00, None)",A,CONSUMER,awe_voltage_reads,['V'],m_2,None
adc65088-66bd-49ba-9deb-a42e538a50a6,awefice,m_3,"[2021-01-01 08:01:00+00:00, None)",A,CONSUMER,awe_voltage_reads,['V'],m_3,None
d351f2b5-3326-45a8-a259-9eaf439da74a,awefice,m_4,"[2021-01-01 08:01:00+00:00, None)",C,CONSUMER,awe_voltage_reads,['V'],m_4,None
b254dc80-07a9-464f-aedf-407450c07a82,awefice,m_5,"[2021-01-01 08:01:00+00:00, None)",C,CONSUMER,awe_voltage_reads,['V'],m_5,None


The `metrics` column is of type text[] and lists the units of measurement for time-series that can be retrieved for a given grid element. Please note that it is possible to have multiple metrics associated with the same `grid_element_data_source_id`. The following is an example of pulling out multiple grid element data sources available for the 'awefice' `grid_id` and the 'm_12' `grid_element_id`.

In [12]:
%%sql

SELECT geds.grid_element_data_source_id,
        geds.grid_id,
        geds.grid_element_id,
        metric_key
FROM grid_element_data_source geds
JOIN UNNEST(geds.metrics::TEXT[]) AS metric_key
    ON true
WHERE grid_id = 'awefice'
    AND grid_element_id = 'm_12';

 * postgresql://


2 rows affected.


grid_element_data_source_id,grid_id,grid_element_id,metric_key
96ecaf9c-4c66-419d-a711-9d9027ff5f28,awefice,m_12,V
e3434ab9-c1aa-4d59-af40-e06efb196724,awefice,m_12,kWh


The `provider` column represents the origin of the stored data, and may be used to properly identify the grid_element_data_source for the time series of interest.

In [13]:
%%sql

SELECT distinct provider
FROM grid_element_data_source;

 * postgresql://


20 rows affected.


provider
SMART_meter_data
awe_scada_reads
ohio_residential_meter
SMART_scada_data
ohio_industrial_meter
ohio_scada_reads
ohio_pv_meter_reading_up
awe_photovoltaic
ohio_commercial_meter
ohio_photovoltaic


---

## Functions

**grid_get_downstream()**

The `grid_get_downstream()` function is one of the tracing functions available in EDM, and is mentioned here to demonstrate a function with grid and grid element arguments. Its input arguments can be found by running the following query.

Please refer to the [grid_tracing.ipynb](../3_grid_tracing/grid_tracing.ipynb) notebook for examples of all tracing functions.


In [14]:
%%sql

SELECT function_args
FROM get_function_documentation('grid_get_downstream');

 * postgresql://


1 rows affected.


function_args
"grid_id text, grid_element_id text, include_self boolean DEFAULT true"


Below is an example with the 'awefice' `grid_id`, the 'm_10' `grid_element_id` and False to exclude the specified `grid_element_id` from the output list.

In [15]:
%%sql

SELECT *
FROM grid_get_downstream('awefice', 'm_10', False);

 * postgresql://


3 rows affected.


grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,meta
awefice,batt_10,Battery,battery,ABC,None,True,True,False,None,con_135_t2,VCN.batt_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': 'batt_10', 'label': 'batt_10', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_10', 'longitude': None, 'lv_feeder': None, 'mv_feeder': None, 'ownership': 'private', 'make_model': 'LG CHEM RESU10H', 'enclosure_id': None, 'voltage_level': 'LV', 'capacity_usable': 9.6, 'commission_date': None, 'capacity_nominal': 9.8, 'manufacture_date': None, 'efficiency_round_trip': 97.96, 'max_peak_charge_power': 7.0, 'parent_transformer_id': 'transformer_36', 'max_peak_discharge_power': 7.0, 'max_continuous_charge_power': 5.0, 'max_continuous_discharge_power': 5.0}"
awefice,evch_10,EVCharger,ev_charger,ABC,None,True,True,False,None,con_135_t2,VCN.evch_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': None, 'label': 'evch_10', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_10', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'ChargePoint Home Flex [CPH50-NEMA14-50-L23]', 'rating_kva': '', 'max_current': 32, 'max_voltage': 240, 'is_dc_charger': False, 'voltage_level': 'LV', 'charging_level': 'level_2', 'connector_type': 'NEMA 14-30', 'active_charging_power': 7680, 'parent_transformer_id': 'transformer_36', 'active_generation_power': 0}"
awefice,pv_10,Photovoltaic,photovoltaic,ABC,None,True,True,False,None,con_135_t2,VCN.pv_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': 'pv_10', 'label': 'pv_10', 'sdp_id': None, 'ac_size': 3651, 'dc_size': 4440, 'racking': 'roof', 'latitude': None, 'meter_id': 'm_10', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'panel_area': 1.7272, 'tilt_angle': 42, 'azimuth_angle': 170, 'voltage_level': 'LV', 'dc_to_ac_ratio': 1.216, 'commission_date': '2014-08-16', 'number_of_panels': 12, 'panel_make_model': 'LG Neon R [LG370Q1C-V5]', 'panel_rated_power': 370.0, 'degradation_annual': 0.3, 'generation_capacity': 3651, 'inverter_make_model': 'SolarEdge HDWave', 'degradation_25_years': 7.5, 'parent_transformer_id': 'transformer_36'}"


**ts_data_source_select()**

The `ts_data_source_select()` returns time series data for the given `id`, `metric_key`, and `timerange` input arguments.

Please note that the `id` argument refers to the `grid_element_data_source_id` from the `grid_element_data_source` table discussed in the above. And the `metric_key` is one of the metrics listed in the `metrics` column in the same table. Below is an example of retrieving V time series for the 'm_12' grid_element (whose grid_element_data_source_id is '96ecaf9c-4c66-419d-a711-9d9027ff5f28') for April 1 ~ 30, 2022 time range.

In [16]:
%%sql

SELECT timestamp, value
FROM ts_data_source_select('96ecaf9c-4c66-419d-a711-9d9027ff5f28', 'V', '[2022-04-01, 2022-04-30]'
                          )
LIMIT 5;

 * postgresql://


5 rows affected.


timestamp,value
2022-04-01 00:00:00+00:00,488.35
2022-04-01 01:00:00+00:00,474.3
2022-04-01 02:00:00+00:00,488.22
2022-04-01 03:00:00+00:00,489.39
2022-04-01 04:00:00+00:00,484.31


More generally, time series for multiple grid elements and metrics over the entire period can be retrieved by joining multiple tables as the following. Please note that the query output is ordered and limited for demonstration purposes.

In [17]:
%%sql

SELECT geds.grid_element_id,
        geds.type,
        metric_key,
        tdss.timestamp,
        tdss.value
FROM grid_element ge
JOIN grid_element_data_source geds
    ON geds.grid_id = ge.grid_id
    AND geds.grid_element_id = ge.grid_element_id
JOIN UNNEST(geds.metrics::TEXT[]) metric_key
    ON true
JOIN ts_data_source_select(geds.grid_element_data_source_id, metric_key) tdss
    ON true
WHERE ge.grid_id = 'awefice'
    AND ge.type = 'Meter'
    AND geds.type = 'CONSUMER'
ORDER by 3
LIMIT 5;

 * postgresql://


5 rows affected.


grid_element_id,type,metric_key,timestamp,value
m_4,CONSUMER,kWh,2021-01-01 10:00:00+00:00,1.774
m_4,CONSUMER,kWh,2021-01-01 11:00:00+00:00,1.431
m_4,CONSUMER,kWh,2021-01-01 08:00:00+00:00,1.889
m_4,CONSUMER,kWh,2021-01-01 09:00:00+00:00,1.585
m_4,CONSUMER,kWh,2021-01-01 12:00:00+00:00,1.741


---

## SQL & Python

**Data Retrieval**

Below are a few ways to save SQL output into a local Python variable.

Option 1

In [18]:
%%sql result <<

SELECT grid_element_id, type
FROM grid_element
WHERE grid_id = 'awefice';

 * postgresql://


200 rows affected.
Returning data to local variable result


Option 2

In [19]:
# Save the sql output as a variable in one line.
result = %sql SELECT grid_element_id, type FROM grid_element WHERE grid_id = 'awefice'

 * postgresql://


200 rows affected.


Option 3

In [20]:
# Save the sql output as a variable in multiple lines.
result = %sql SELECT grid_element_id, type \
                FROM grid_element \
                WHERE grid_id = 'awefice'

 * postgresql://


200 rows affected.


Option 4

In [21]:
# Use the %sql magic command with a variable holding the query.
# Define your query.
query = """
SELECT grid_element_id, type
FROM grid_element
WHERE grid_id = 'awefice';
"""

# Execute the query using %sql magic command.
result = %sql $query

 * postgresql://


200 rows affected.


Then the following code can be run to convert the above SQL output `result` into a Python dataframe to work with.

In [22]:
df = result.DataFrame()

# Return the first three lines of the dataframe to take a peek.
df.head(3)

,grid_element_id,type
0,batt_10,Battery
1,batt_19,Battery
2,batt_24,Battery


**Variable Input**

A python variable can be integrated into a SQL query as a **named parameter**: reference it with a leading colon (e.g. `:grid_element_id`). ipython-sql binds the value safely at execution time, so there is no need to add quotes yourself.

In [23]:
grid_element_id = "m_12"

In [24]:
%%sql

SELECT *
FROM grid_element_data_source
WHERE grid_element_id = :grid_element_id;

 * postgresql://


2 rows affected.


grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction
96ecaf9c-4c66-419d-a711-9d9027ff5f28,awefice,m_12,"[2021-01-01 08:01:00+00:00, None)",ABC,CONSUMER,awe_voltage_reads,['V'],m_12,None
e3434ab9-c1aa-4d59-af40-e06efb196724,awefice,m_12,"[2021-01-01 08:00:00+00:00, None)",ABC,CONSUMER,awe_meter_reading,['kWh'],m_12,None


In [25]:
# Named parameters also work when the query is stored in a variable.
query = """
SELECT *
FROM grid_element_data_source
WHERE grid_element_id = :grid_element_id;
"""

%sql $query

 * postgresql://


2 rows affected.


grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction
96ecaf9c-4c66-419d-a711-9d9027ff5f28,awefice,m_12,"[2021-01-01 08:01:00+00:00, None)",ABC,CONSUMER,awe_voltage_reads,['V'],m_12,None
e3434ab9-c1aa-4d59-af40-e06efb196724,awefice,m_12,"[2021-01-01 08:00:00+00:00, None)",ABC,CONSUMER,awe_meter_reading,['kWh'],m_12,None


In [26]:
# Notice that when including properties from the meta column using this SQL option, there should be no space between `meta` and `->>`.
query = """
SELECT grid_element_id,
    meta->> 'meter_number' as meter_number,
    meta->> 'voltage_level' as voltage_level
FROM grid_element
WHERE grid_id = 'awefice'
    AND grid_element_id = :grid_element_id;
"""

%sql $query

 * postgresql://


1 rows affected.


grid_element_id,meter_number,voltage_level
m_12,M7-2558866S,LV
